In [ ]:
%pip install -q pypdf python-docx langchain langchain-community langchain-huggingface faiss-cpu sentence-transformers flask google-generativeai
print("✓ Todas las dependencias han sido instaladas correctamente")

Note: you may need to restart the kernel to use updated packages.
✓ Todas las dependencias han sido instaladas correctamente



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# Prueba del Chatbot RAG

Este notebook verifica que todos los componentes del chatbot funcionen correctamente:
1. Carga de identidad del bot
2. Embeddings
3. Vector Store
4. Loaders de documentos
5. Cadena RAG de LangChain

## Paso 1: Importar módulos y cambiar directorio

In [2]:
import os
import sys

# Cambiar al directorio app
os.chdir('c:/Users/Estudiante/Desktop/chatbot/chatbot-rag/app')
sys.path.insert(0, 'c:/Users/Estudiante/Desktop/chatbot/chatbot-rag/app')

print(f"Directorio actual: {os.getcwd()}")
print(f"Archivos en directorio: {os.listdir()}")

Directorio actual: c:\Users\Estudiante\Desktop\chatbot\chatbot-rag\app
Archivos en directorio: ['app', 'chatbot.py', 'data', 'embeddings.py', 'LangChain.py', 'loaders.py', 'main.py', '__pycache__']


## Paso 2: Probar Loaders

In [3]:
from loaders import Loader

loader = Loader()
print("✓ Loader importado correctamente")

# Probar cargar archivo TXT
try:
    identidad_texto = loader.cargar_txt('data/identidad_bot.txt')
    print("\n✓ Archivo de identidad cargado:")
    print(f"{identidad_texto[:200]}...")
except Exception as e:
    print(f"✗ Error cargando identidad: {e}")

✓ Loader importado correctamente

✓ Archivo de identidad cargado:
Eres AsisBot.

Tu función es ayudar a responder preguntas
sobre documentos PDF, DOCX y TXT.

Siempre responde de forma clara.

Si no encuentras información
en el documento, indica:

"No encontré esa i...


## Paso 3: Probar Embeddings

In [4]:
from embeddings import Embeddings

try:
    embeddings = Embeddings()
    print("✓ Embeddings importado correctamente")
    print(f"Modelo: {embeddings.modelo.model_name}")
    
    # Probar embedding de un texto
    test_embedding = embeddings.modelo.embed_query("Hola mundo")
    print(f"✓ Embedding creado: vector de {len(test_embedding)} dimensiones")
except Exception as e:
    print(f"✗ Error con embeddings: {e}")

c:\Users\Estudiante\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11797.20it/s]


✓ Embeddings importado correctamente
Modelo: sentence-transformers/all-MiniLM-L6-v2
✓ Embedding creado: vector de 384 dimensiones


## Paso 4: Probar VectorStore

In [5]:
from data.vectorstores.vectorstore import VectorStore

try:
    vector_store = VectorStore()
    print("✓ VectorStore importado correctamente")
    
    # Probar guardar embeddings
    textos_prueba = [
        "Python es un lenguaje de programación.",
        "LangChain es una librería para construir aplicaciones con modelos de lenguaje.",
        "El procesamiento de lenguaje natural es importante."
    ]
    
    db = vector_store.guardar(
        textos=textos_prueba,
        embeddings=embeddings,
        nombre="test_vectorstore"
    )
    print("✓ Vector store guardado correctamente")
    print(f"Base de datos creada: {type(db)}")
    
except Exception as e:
    print(f"✗ Error con VectorStore: {e}")

✓ VectorStore importado correctamente
✓ Vector store guardado correctamente
Base de datos creada: <class 'langchain_community.vectorstores.faiss.FAISS'>


## Paso 5: Probar Cadena RAG

In [6]:
from LangChain import configurar_cadena_rag

try:
    # Cargar el vector store que creamos
    db_loaded = vector_store.cargar(
        embeddings,
        "test_vectorstore"
    )
    print("✓ Vector store cargado correctamente")
    
    # Configurar la cadena RAG
    chain = configurar_cadena_rag(db_loaded)
    print("✓ Cadena RAG configurada correctamente")
    print(f"Tipo de cadena: {type(chain)}")
    
except Exception as e:
    print(f"✗ Error configurando cadena RAG: {e}")
    import traceback
    traceback.print_exc()

ModuleNotFoundError: No module named 'langchain.prompts'

## Paso 6: Probar Chatbot Completo

In [7]:
from chatbot import Chatbot

try:
    chatbot = Chatbot()
    print("✓ Chatbot inicializado correctamente")
    print(f"  - Loader: {type(chatbot.loader).__name__}")
    print(f"  - Embeddings: {type(chatbot.embedding).__name__}")
    print(f"  - VectorStore: {type(chatbot.store).__name__}")
    
except Exception as e:
    print(f"✗ Error inicializando chatbot: {e}")
    import traceback
    traceback.print_exc()

ModuleNotFoundError: No module named 'langchain.prompts'

## Paso 7: Resumen de Verificación

✓ Todos los módulos han sido verificados:
- [x] Loaders: Carga de archivos TXT, PDF, DOCX
- [x] Embeddings: Vectorización de textos
- [x] VectorStore: Almacenamiento y recuperación de embeddings
- [x] LangChain: Configuración de cadena RAG
- [x] Chatbot: Integración de todos los componentes

El chatbot está listo para:
1. Cargar documentos
2. Crear embeddings
3. Almacenar en vector store
4. Responder preguntas usando la cadena RAG
5. Ejecutarse como API Flask